In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import joblib
import sys
sys.path.append('..')
from app.preprocess import clean_arabic_text

In [ ]:
df = pd.read_csv('../app/data/arabic_dataset_classifiction.csv')
print(f"Dataset chargé: {len(df)} exemples")

df['clean'] = df['text'].apply(clean_arabic_text)
print("\nAperçu:")
print(df[['text', 'clean']].head())
print(df.columns)

In [ ]:
X = df['clean']
y = df['targe']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=1000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Train: {X_train_vec.shape[0]} samples")
print(f"Test: {X_test_vec.shape[0]} samples")

In [ ]:
modeles = {
    'Naive Bayes': MultinomialNB(),
    'SVM': SVC(kernel='linear', probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Decision Tree': DecisionTreeClassifier(max_depth=20),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Descent': SGDClassifier(loss='modified_huber', max_iter=1000)
}

resultats = []
best_f1 = 0
best_modele = None
best_nom = None

print("="*50)
print("COMPARAISON DES MODÈLES")
print("="*50)

for nom, modele in modeles.items():
    modele.fit(X_train_vec, y_train)
    y_pred = modele.predict(X_test_vec)
    
    f1 = f1_score(y_test, y_pred, average='weighted')
    acc = accuracy_score(y_test, y_pred)
    
    resultats.append({
        'Modèle': nom,
        'Accuracy': round(acc, 4),
        'F1-Score': round(f1, 4)
    })
    
    print(f"\n{nom}")
    print(f"    Accuracy: {acc:.4f}")
    print(f"    F1-Score: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_modele = modele
        best_nom = nom

In [ ]:
print("\n" + "="*50)
print(" RÉSULTATS FINAUX - CLASSEMENT")
print("="*50)

resultats_df = pd.DataFrame(resultats)
resultats_df = resultats_df.sort_values('F1-Score', ascending=False)

for i, row in resultats_df.iterrows():
    print(f"{row['Modèle']:20} | F1: {row['F1-Score']:.4f} | Acc: {row['Accuracy']:.4f}")

print(f"\nMeilleur modèle: {best_nom} avec F1 = {best_f1:.4f}")

In [ ]:
joblib.dump(best_modele, '../app/models/best_model.pkl')
joblib.dump(vectorizer, '../app/models/vectorizer.pkl')

In [ ]:
model_path = '../app/models/best_model.pkl'

try:
    loaded_model = joblib.load(model_path)
    print(f"Modèle chargé avec succès depuis : {model_path}")
    
    test_samples = [
        "انتهت المواجهة الختامية بنصر مغربي ثمين، بفضل الاستراتيجية الدفاعية المحكمة والسيطرة المطلقة على مجريات اللعب",
        "استقبل رئيس الحكومة صباح اليوم وفدا ديبلوماسيا رفيع المستوى لبحث سبل تعزيز التعاون الثنائي ومناقشة قضايا الأمن والسلم في المنطقة العربية والعالم",
        "أعلنت وزارة المالية عن حزمة جديدة من الإجراءات التحفيزية لدعم الشركات الناشئة وجذب الاستثمارات الأجنبية بعد تراجع معدلات التضخم واستقرار سعر صرف العملة الوطنية",
        "ينظم المعهد الثقافي ندوة فكرية حول دور الأدب العربي في تعزيز الحوار الحضاري بمشاركة نخبة من الكتاب والشعراء والفنانين التشكيليين من مختلف أنحاء العالم",
        "أكدت الدراسات العلمية الحديثة أن الحفاظ على البيئة وتقليل انبعاثات الكربون يساهم بشكل كبير في تحسين جودة الحياة والحد من مخاطر التغير المناخي المستقبلي"
    ]
    
    mapping = {0: "Culture", 1: "Diverse", 2: "Economy", 3: "Politic", 4: "Sport"}
    
    for test in test_samples:
        cleaned = clean_arabic_text(test)
        print(f"\nTest: '{test}'")
        
        vec = vectorizer.transform([cleaned])
        probabilities = loaded_model.predict_proba(vec)[0]
        
        for emotion, prob in zip(mapping.values(), probabilities):
            print(f"Probabilité de {emotion}: {prob:.4f}")
        print("*"*30)
except FileNotFoundError:
    print(f"Erreur : Le fichier {model_path} est introuvable.")
except Exception as e:
    print(f"Erreur : {e}")